## deps

In [1]:
!pip install tqdm fastparquet
!pip install torch transformers datasets accelerate scikit-learn

## run

In [1]:
import requests
import yaml
import getpass
# import zstandard as zstd
import pandas as pd
import json
import io

from typing import Dict, List, Any
from tqdm import tqdm

In [2]:
df_queries = pd.read_parquet('esci-data/shopping_queries_dataset/shopping_queries_dataset_examples.parquet')
df_queries = df_queries[df_queries["product_locale"] == "us"]


In [3]:
df_queries.head(5)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
0,0,revent 80 cfm,0,B000MOO21W,us,I,0,1,train
1,1,revent 80 cfm,0,B07X3Y6B1V,us,E,0,1,train
2,2,revent 80 cfm,0,B07WDM7MQQ,us,E,0,1,train
3,3,revent 80 cfm,0,B07RH6Z8KW,us,E,0,1,train
4,4,revent 80 cfm,0,B07QJ7WYFQ,us,E,0,1,train


In [9]:
df_products = pd.read_parquet('esci-data/shopping_queries_dataset/shopping_queries_dataset_products.parquet')
df_products = df_products[df_products["product_locale"] == "us"]
df_products.head(5)

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
167168,B003O0MNGC,Delta BreezSignature VFB25ACH 80 CFM Exhaust B...,NaN,Virtually silent at less than 0.3 sones\nPreci...,DELTA ELECTRONICS (AMERICAS) LTD.,White,us
167169,B00MARNO5Y,Aero Pure AP80RVLW Super Quiet 80 CFM Recessed...,NaN,Super quiet 80CFM energy efficient fan virtual...,Aero Pure,White,us
167170,B011RX6PNO,Aero Pure AP120H-SL W Slim Fit 120 CFM Bathroo...,NaN,"Slim Fit Housing Fits Into 2"" X 6"" Ceiling Joi...",Aero Pure,White Finish,us
167171,B01MZIK0PI,Delta Electronics (Americas) Ltd. RAD80 Delta ...,NaN,Quiet operation at 1.5 Sones\nPrecision engine...,DELTA ELECTRONICS (AMERICAS) LTD.,With Heater,us
167172,B01N5Y6002,Delta Electronics (Americas) Ltd. GBR80HLED De...,NaN,Ultra energy-efficient LED module (11-watt equ...,DELTA ELECTRONICS (AMERICAS) LTD.,"With LED Light, Dual Speed & Humidity Sensor",us


In [10]:
df = pd.merge(
    df_queries,
    df_products,
    how='left',
    left_on=['product_locale','product_id'],
    right_on=['product_locale', 'product_id']
)
df.head(5)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split,product_title,product_description,product_bullet_point,product_brand,product_color
0,0,revent 80 cfm,0,B000MOO21W,us,I,0,1,train,Panasonic FV-20VQ3 WhisperCeiling 190 CFM Ceil...,NaN,WhisperCeiling fans feature a totally enclosed...,Panasonic,White
1,1,revent 80 cfm,0,B07X3Y6B1V,us,E,0,1,train,Homewerks 7141-80 Bathroom Fan Integrated LED ...,NaN,OUTSTANDING PERFORMANCE: This Homewerk's bath ...,Homewerks,80 CFM
2,2,revent 80 cfm,0,B07WDM7MQQ,us,E,0,1,train,Homewerks 7140-80 Bathroom Fan Ceiling Mount E...,NaN,OUTSTANDING PERFORMANCE: This Homewerk's bath ...,Homewerks,White
3,3,revent 80 cfm,0,B07RH6Z8KW,us,E,0,1,train,Delta Electronics RAD80L BreezRadiance 80 CFM ...,This pre-owned or refurbished product has been...,Quiet operation at 1.5 sones\nBuilt-in thermos...,DELTA ELECTRONICS (AMERICAS) LTD.,White
4,4,revent 80 cfm,0,B07QJ7WYFQ,us,E,0,1,train,Panasonic FV-08VRE2 Ventilation Fan with Reces...,NaN,The design solution for Fan/light combinations...,Panasonic,White


In [11]:
len(df)

1818825

In [25]:
ESCI_RATING = {"I": 0, "C": 1, "S": 2, "E": 3}

## train student

In [17]:
df[['query', 'esci_label', 'product_title', 'product_description', 'product_color', 'product_brand']].head(5)

,query,esci_label,product_title,product_description,product_color,product_brand
0,revent 80 cfm,I,Panasonic FV-20VQ3 WhisperCeiling 190 CFM Ceil...,NaN,White,Panasonic
1,revent 80 cfm,E,Homewerks 7141-80 Bathroom Fan Integrated LED ...,NaN,80 CFM,Homewerks
2,revent 80 cfm,E,Homewerks 7140-80 Bathroom Fan Ceiling Mount E...,NaN,White,Homewerks
3,revent 80 cfm,E,Delta Electronics RAD80L BreezRadiance 80 CFM ...,This pre-owned or refurbished product has been...,White,DELTA ELECTRONICS (AMERICAS) LTD.
4,revent 80 cfm,E,Panasonic FV-08VRE2 Ventilation Fan with Reces...,NaN,White,Panasonic


In [19]:
df = df[df['esci_label'].notna()]
df.head()

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split,product_title,product_description,product_bullet_point,product_brand,product_color
0,0,revent 80 cfm,0,B000MOO21W,us,I,0,1,train,Panasonic FV-20VQ3 WhisperCeiling 190 CFM Ceil...,NaN,WhisperCeiling fans feature a totally enclosed...,Panasonic,White
1,1,revent 80 cfm,0,B07X3Y6B1V,us,E,0,1,train,Homewerks 7141-80 Bathroom Fan Integrated LED ...,NaN,OUTSTANDING PERFORMANCE: This Homewerk's bath ...,Homewerks,80 CFM
2,2,revent 80 cfm,0,B07WDM7MQQ,us,E,0,1,train,Homewerks 7140-80 Bathroom Fan Ceiling Mount E...,NaN,OUTSTANDING PERFORMANCE: This Homewerk's bath ...,Homewerks,White
3,3,revent 80 cfm,0,B07RH6Z8KW,us,E,0,1,train,Delta Electronics RAD80L BreezRadiance 80 CFM ...,This pre-owned or refurbished product has been...,Quiet operation at 1.5 sones\nBuilt-in thermos...,DELTA ELECTRONICS (AMERICAS) LTD.,White
4,4,revent 80 cfm,0,B07QJ7WYFQ,us,E,0,1,train,Panasonic FV-08VRE2 Ventilation Fan with Reces...,NaN,The design solution for Fan/light combinations...,Panasonic,White


In [26]:
with open('training_data.jsonl', 'w') as fh:
    for r in tqdm(df.itertuples(), total=len(df)):
        fh.write(json.dumps({"query": r.query,
                             "product_text": f'{r.product_title}\n{r.product_description}\n{r.product_brand}\n{r.product_color}',
                             "label": ESCI_RATING.get(r.esci_label)}) + "\n")


100%|██████████████████████████████████████████████████████| 1818825/1818825 [00:09<00:00, 195950.24it/s]


Python(5994) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.3/127.3 MB 37.2 MB/s  0:00:03 38.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 37.7 MB/s  0:00:00.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 32.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 34.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 34.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 27.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 36.4 MB/s  0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 33.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 37.8 MB/s  0:00:000.2 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 26.6 MB/s  0:00:000.1 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 25.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 27.6 MB/s  0:

In [27]:
import numpy as np

from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    f1_score,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

/Users/frutik/work/judge-student/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [28]:
MODEL_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
OUTPUT_DIR = "./amazon-ecommerce-judge"

In [29]:
ID_TO_LABEL = {
    0: "Irrelevant",
    1: "Complement",
    2: "Substitute",
    3: "Exact",
}


LABEL_TO_ID = {
    label: label_id
    for label_id, label in ID_TO_LABEL.items()
}

In [30]:
# Load JSONL.
dataset = load_dataset(
    "json",
    data_files="training_data.jsonl",
    split="train",
)

Generating train split: 1818825 examples [00:04, 390138.42 examples/s]


In [31]:
from datasets import ClassLabel

label_feature = ClassLabel(
    names=list(ID_TO_LABEL.values())
)

dataset = dataset.cast_column("label", label_feature)

dataset = dataset.train_test_split(
    test_size=0.15,
    seed=42,
    stratify_by_column="label",
)

Casting the dataset: 100%|█████████████████████████| 1818825/1818825 [00:00<00:00, 3326064.59 examples/s]


In [32]:
from collections import Counter

print(Counter(dataset["train"]["label"]))
print(Counter(dataset["test"]["label"]))

Counter({3: 1060424, 2: 313916, 0: 137619, 1: 34042})
Counter({3: 187134, 2: 55397, 0: 24286, 1: 6007})


In [33]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize(batch):
    return tokenizer(
        batch["query"],
        batch["product_text"],
        truncation="only_second",
        max_length=256,
    )

In [34]:
print([
    len(ID_TO_LABEL) == 4,
    len(LABEL_TO_ID) == 4
])

[True, True]


In [35]:
print([ID_TO_LABEL, LABEL_TO_ID])

[{0: 'Irrelevant', 1: 'Complement', 2: 'Substitute', 3: 'Exact'}, {'Irrelevant': 0, 'Complement': 1, 'Substitute': 2, 'Exact': 3}]


In [36]:
tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["query", "product_text"],
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,

    # The original reranker has a one-score output head.
    # We replace it with a new randomly initialized three-class head.
    ignore_mismatched_sizes=True,
)

Map: 100%|█████████████████████████████████████████████| 272824/272824 [00:11<00:00, 23934.12 examples/s]
[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1`.
Loading weights: 100%|██████████████████████████████████████████████| 201/201 [00:00<00:00, 10362.54it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                        | Status   |                                                                                       
---------------------------+----------+---------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1]) vs model:torch.Size([4])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1, 384]) vs model:torch.Size([4, 384])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match

In [37]:
print(model.config.num_labels)
print(model.config.id2label)
print(model.config.label2id)

4
{0: 'Irrelevant', 1: 'Complement', 2: 'Substitute', 3: 'Exact'}
{'Irrelevant': 0, 'Complement': 1, 'Substitute': 2, 'Exact': 3}


In [38]:
def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro",
        ),
        "weighted_f1": f1_score(
            labels,
            predictions,
            average="weighted",
        ),
        "weighted_kappa": cohen_kappa_score(
            labels,
            predictions,
            weights="quadratic",
        ),
    }

In [39]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,

    # Suitable for your RTX 4080.
    fp16=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none",
    seed=42,
)

In [40]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

/Users/frutik/work/judge-student/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(trainer.evaluate())

Check the actual class performance:

In [111]:
import numpy as np

from sklearn.metrics import classification_report, confusion_matrix

prediction_output = trainer.predict(tokenized_dataset["test"])

y_true = prediction_output.label_ids
y_pred = np.argmax(prediction_output.predictions, axis=1)

class_names = [
    ID_TO_LABEL[i]
    for i in range(len(ID_TO_LABEL))
]

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=3,
        zero_division=0,
    )
)

print(
    confusion_matrix(
        y_true,
        y_pred,
    )
)

/Users/frutik/work/judge-student/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

  Irrelevant      0.798     0.878     0.836       597
  Complement      0.160     0.121     0.138        33
  Substitute      0.000     0.000     0.000       159
       Exact      0.756     0.896     0.820       578

    accuracy                          0.765      1367
   macro avg      0.428     0.474     0.448      1367
weighted avg      0.672     0.765     0.715      1367

[[524   5   0  68]
 [ 11   4   0  18]
 [ 66  12   0  81]
 [ 56   4   0 518]]
